# G10 — Do fractal encoders carry a frozen caption head? (recoverable structure)

G9 measured **visible** structure: raw Spearman shape agreement, no map fitted. It found the
fractal CNN at the pixel floor (highly anisotropic), rescued to a small residual only after
whitening. That answers *is the structure visible?* — **not** *is it recoverable?*

This notebook runs the Atlas test of **recoverable** structure, the C.13 / G4 / E.2 protocol:

    fractal encoder --W_entry--> FROZEN 512-d hub --FROZEN caption head--> bge space --retrieval R@1

A **fitted linear entry map** (ridge) pushes the fractal features into the frozen hub; the
frozen caption head (trained on DINOv2-small, never on fractals) reads them out; retrieval R@1
is scored on held-out rows and reported as **% of a head fitted natively on the fractal encoder**.

**Why this can succeed where G9 read a floor.** Shape agreement measures what is visible without
fitting; a fitted entry map can recover a linear correspondence that raw geometry does not show —
this is exactly the E.6 / E.12 lesson (ConvNeXt: lowest raw ρ, highest transfer). So the fractal
CNN could be near-zero in G9 yet non-trivial here. That is the whole point of running it.

**Canonical protocol (E.1 — the four-space hub that reproduces the published figures):**
DINOv2 small/base/large + bge-m3, each scaled by its **mean column std**, whitened-PCA to
**512-d** on **8,533** train rows, random **seed-0** split, **1,000** held out, head = ridge
(α 1e-2) into **bge-m3**, entry map α = 1.0.

**Reproduction gate (must pass before any fractal number is trusted):** img_base **95.9%**,
img_large **92.9%**, and the random-map control at chance. If it misses, every fractal verdict is
stamped **UNVERIFIED**.

**Baselines for the fractal encoders**, same protocol: same-architecture **random-init** and
**ImageNet-1k** twins, a **32×32 pixel** floor, and a **native** head fitted on the fractal
encoder itself (the denominator).

**Pre-registered (fixed before any fractal entry map is fit):**
| # | prediction | threshold |
|---|---|---|
| **H1** | fractal transfer > random-twin and > pixel, both architectures | > 2 combined bootstrap SEs, held-out |
| **H2** | fractal transfer < ImageNet-twin transfer | expected; the informative quantity is the % |
| **H3** | recoverable % ≫ G9's visible reality-fraction R | i.e. a fitted map recovers more than raw ρ shows |

Run `SYNTHETIC = True` for a 2-min dry run of every cell; `False` for the real run.

## 1 — Install & config

In [ ]:
import importlib, importlib.util, subprocess, sys
for pkg, mod in [("timm", "timm"), ("gdown", "gdown")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import os, json, time
from pathlib import Path
import numpy as np, pandas as pd

SYNTHETIC = False
DIM       = 512          # hub width (the published operating point; below the 768 cliff)
ALPHA_HEAD  = 1e-2
ALPHA_ENTRY = 1.0
N_ITEMS   = 9533
SEED      = 0
N_BOOT    = 200          # bootstrap resamples of held-out rows for SEs
GAP_K     = 2.0
# A native-head R@1 at or below this is 'at the floor': the encoder recovers essentially
# nothing, so '% of native' is a ratio of two floor-level numbers and is MEANINGLESS. The
# guard is the max of an absolute floor and 2x the random-map control (set after the gate).
TRANSFER_FLOOR = 0.05

if SYNTHETIC:
    N_ITEMS = 4200          # keep rows/dim realistic (train ~3780 vs max dim 2048) but fit in a small box
    N_BOOT = 60
    DATA_DIR = Path("/tmp/g10_syn"); WORK = Path("/tmp/g10_work")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False); base = Path("/content/drive/MyDrive")
    except Exception:
        base = Path(os.environ.get("DRIVE_ROOT", "."))
    DATA_DIR = Path(os.environ.get("DATA_DIR", base / "convergence_experiment"))
    if not DATA_DIR.exists():
        hits = [p for p in base.glob("**/convergence_experiment") if p.is_dir()][:1]
        DATA_DIR = hits[0] if hits else DATA_DIR
    WORK = Path("/content/g10_work")
G9_OUT = DATA_DIR / "G9_fractal"        # where G9 wrote its fractal feature caches
OUT = DATA_DIR / "G10_fractal_hub"
for p in (WORK, OUT): p.mkdir(parents=True, exist_ok=True)
print(f"SYNTHETIC={SYNTHETIC}\nDATA_DIR={DATA_DIR} exists={DATA_DIR.exists()}\nG9_OUT={G9_OUT} exists={G9_OUT.exists()}\nOUT={OUT}")

## 2 — Hub core (numpy; identical math to C.13/G4)

In [ ]:
"""G10 core — the Atlas hub entry-map + frozen caption head test of RECOVERABLE structure.

This is the C.13 / G4 / E.2 protocol, reconstructed exactly (per E.1). It measures what G9 does
NOT: whether a FITTED linear entry map carries a fractal encoder's features into the frozen hub such
that a frozen caption head reads them out — recoverable structure, not merely visible shape agreement.

Canonical protocol (E.1, the four-space hub that reproduces the published figures to the decimal):
  * hub members: DINOv2 small/base/large + bge-m3 (bge is also the head target)
  * per-space scaling: divide each space by its mean column std (NOT row L2)
  * hub basis: whitened PCA of the concatenated scaled spaces, 512-d, fit on 8,533 train rows
  * split: random, seed 0, 1,000 held out
  * head: ridge (alpha 1e-2), hub coords -> bge-m3 space
  * retrieval: cosine, text->image, R@1 on held-out
  * validation gate: img_base ~95.9-96.5%, img_large ~92.9-93.8%, SigLIP ~94.2% of native

Torch-free: hub math is numpy. Feature extraction (fractal encoders) reuses the G9 caches.
"""
import numpy as np
from numpy.linalg import svd, lstsq


# ------------------------------------------------------------------ scaling
def col_std_scale(X, mu=None, sd=None):
    """Per-space scaling used by the original hub: subtract column mean, divide by
    the MEAN column std (a single scalar per space), returning stats so the same
    transform applies to held-out rows and to new encoders."""
    X = np.asarray(X, dtype=np.float64)
    if mu is None:
        mu = X.mean(0, keepdims=True)
    if sd is None:
        sd = (X - mu).std(0).mean()          # scalar: mean column std
        sd = sd if sd > 1e-12 else 1.0
    return (X - mu) / sd, mu, sd


# ------------------------------------------------------------------ hub basis
def fit_hub(scaled_train, dim=512, eps=1e-9):
    """Whitened PCA of the concatenated per-space-scaled TRAIN blocks -> hub basis.
    scaled_train: list of (n_train, d_i) arrays already col_std-scaled.
    Returns (mean, components V[:dim], inv_sqrt_eigs) to project any concat row."""
    C = np.concatenate(scaled_train, axis=1)
    mu = C.mean(0, keepdims=True)
    U, S, Vt = svd(C - mu, full_matrices=False)
    k = min(dim, int((S > eps * S[0]).sum()))
    Vk = Vt[:k]
    inv = np.sqrt(len(C) - 1) / S[:k]        # whitening: unit variance per component
    return dict(mu=mu, Vk=Vk, inv=inv, k=k, dims=[b.shape[1] for b in scaled_train])


def project_hub(hub, scaled_concat_row):
    """Map a concatenated scaled row block (n, sum d_i) into hub coords (n, k)."""
    return ((np.asarray(scaled_concat_row, dtype=np.float64) - hub["mu"]) @ hub["Vk"].T) * hub["inv"]


# ------------------------------------------------------------------ per-encoder entry map
def entry_map(Xtr_scaled, hub_tr, alpha=1.0):
    """One linear map from a SINGLE encoder's scaled features into hub coords,
    fit on train rows (ridge). This is the map slot the random control replaces."""
    d = Xtr_scaled.shape[1]
    A = Xtr_scaled.T @ Xtr_scaled + alpha * np.eye(d)
    B = Xtr_scaled.T @ hub_tr
    W = np.linalg.solve(A, B)                # (d, k)
    return W


# ------------------------------------------------------------------ caption head
def fit_head(hub_tr, Ytr, alpha=1e-2):
    """Ridge from hub coords -> caption (bge) space, on train rows."""
    k = hub_tr.shape[1]
    A = hub_tr.T @ hub_tr + alpha * np.eye(k)
    return np.linalg.solve(A, hub_tr.T @ Ytr)   # (k, d_cap)


# ------------------------------------------------------------------ retrieval
def l2n(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float64)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)


def recall_at_k(pred, gallery, ks=(1, 5, 10)):
    """Text->image style retrieval: row i of pred should retrieve row i of gallery.
    Returns {k: recall}. Diagonal-rank protocol, held-out rows only."""
    P, G = l2n(pred), l2n(gallery)
    S = P @ G.T
    n = S.shape[0]
    order = np.argsort(-S, axis=1)
    ranks = np.empty(n, dtype=int)
    for i in range(n):
        ranks[i] = np.where(order[i] == i)[0][0]
    return {f"R@{k}": float((ranks < k).mean()) for k in ks}


# ------------------------------------------------------------------ full pipeline
def run_transfer(source_scaled, hub, head, Ytr_ignored, tr, ho, cap_target,
                 alpha_entry=1.0, random_control=False, rng=None):
    """Fit an entry map for `source` on train rows, push held-out rows through the
    FROZEN hub and FROZEN head, and score retrieval against the caption gallery.

    source_scaled : (n, d) already col_std-scaled with the SOURCE's own stats
    hub, head     : frozen (fit elsewhere on the hub members)
    cap_target    : (n, d_cap) caption/bge vectors, the retrieval gallery
    """
    Xtr, Xho = source_scaled[tr], source_scaled[ho]
    hub_tr = hub["_train_coords"]            # cached hub coords on train rows
    if random_control:
        rng = np.random.default_rng(0) if rng is None else rng
        W = rng.normal(size=(Xtr.shape[1], hub["k"])) / np.sqrt(Xtr.shape[1])
    else:
        W = entry_map(Xtr, hub_tr, alpha=alpha_entry)
    hub_ho = Xho @ W                          # source -> hub (held-out)
    pred = hub_ho @ head                      # hub -> caption space
    return recall_at_k(pred, cap_target[ho])


def native_head_recall(source_scaled, cap_target, tr, ho, alpha=1e-2):
    """Upper reference: a head fitted DIRECTLY on this encoder's own features
    (no hub), so 'percent of native' is transfer / this."""
    Xtr, Xho = source_scaled[tr], source_scaled[ho]
    k = Xtr.shape[1]
    H = np.linalg.solve(Xtr.T @ Xtr + alpha * np.eye(k), Xtr.T @ cap_target[tr])
    return recall_at_k(Xho @ H, cap_target[ho])

## 3 — Self-test — stop if any FAIL

In [ ]:
from scipy.stats import ortho_group
import numpy as np
from scipy.stats import ortho_group

rng = np.random.default_rng(0)
fails = []
def chk(name, ok, d=""):
    print(f"{'PASS' if ok else 'FAIL'}  {name:<56} {d}")
    if not ok: fails.append(name)

# ---- a shared latent 'reality' and encoders that recover it to varying degree ----
N, L, DCAP = 8000, 16, 48
Z = rng.normal(size=(N, L))                 # latent structure of the world
cap = Z @ rng.normal(size=(L, DCAP)) + 2.0 * rng.normal(size=(N, DCAP))   # bge-like caption space

def encoder(dim, quality, noise, seed):
    r = np.random.default_rng(seed)
    W = r.normal(size=(L, dim))
    return quality * (Z @ W) + noise * r.normal(size=(N, dim))

# four hub members (like DINOv2 s/b/l + bge), varying quality
members = {
    "img_small": encoder(384, 0.45, 2.6, 1),
    "img_base":  encoder(768, 0.70, 2.2, 2),
    "img_large": encoder(1024, 0.95, 1.8, 3),
    "bge":       cap.copy(),                 # bge is a hub member AND the target
}
tr = np.sort(rng.choice(N, 7000, replace=False))
ho = np.sort(np.setdiff1d(np.arange(N), tr))

# scale each space by its own col-std, fit hub on TRAIN concat
scaled, stats = {}, {}
for k, X in members.items():
    s, mu, sd = col_std_scale(X)
    scaled[k], stats[k] = s, (mu, sd)
hub = fit_hub([scaled[k][tr] for k in members], dim=512)
concat_tr = np.concatenate([scaled[k][tr] for k in members], axis=1)
hub["_train_coords"] = project_hub(hub, concat_tr)   # joint coords (used only to fit entry maps)
chk("hub dim <= 512 and <= train rank", hub["k"] <= 512 and hub["k"] <= len(tr))

# REAL protocol: each encoder gets its OWN entry map into the hub (fit on train), and the head
# is trained on the hub image of ONE encoder (img_small), then applied unchanged to others.
Wmaps = {k: entry_map(scaled[k][tr], hub["_train_coords"], alpha=1.0) for k in members}
small_hub_tr = scaled["img_small"][tr] @ Wmaps["img_small"]
cov = np.cov(small_hub_tr.T)
chk("entry-mapped hub coords have positive variance (train)", np.diag(cov).mean() > 0.05,
    f"mean diag {np.diag(cov).mean():.3f}")
head = fit_head(small_hub_tr, cap[tr], alpha=1e-2)

def native_pct(src):
    t = run_transfer(scaled[src], hub, head, None, tr, ho, cap, alpha_entry=1.0)
    nat = native_head_recall(scaled[src], cap, tr, ho)
    return t["R@1"], nat["R@1"], (t["R@1"] / nat["R@1"] if nat["R@1"] > 0 else np.nan)

# transfer should RISE with encoder quality (small < base < large)
r_small = run_transfer(scaled["img_small"], hub, head, None, tr, ho, cap)["R@1"]
r_base  = run_transfer(scaled["img_base"],  hub, head, None, tr, ho, cap)["R@1"]
r_large = run_transfer(scaled["img_large"], hub, head, None, tr, ho, cap)["R@1"]
print(f"  transfer R@1: small {r_small:.3f}  base {r_base:.3f}  large {r_large:.3f}")
chk("transfer rises (or ties) with encoder quality", r_small <= r_base + 0.03 and r_base <= r_large + 0.03,
    f"{r_small:.3f} <= {r_base:.3f} <= {r_large:.3f}")

# random-map control must collapse
r_ctrl = run_transfer(scaled["img_base"], hub, head, None, tr, ho, cap, random_control=True)["R@1"]
chk("random-map control near chance", r_ctrl < 0.05, f"ctrl R@1 {r_ctrl:.3f}")

# native head >= transfer (native is the upper reference)
for k in ("img_base", "img_large"):
    t, nat, pct = native_pct(k)
    print(f"  {k}: transfer {t:.3f}  native {nat:.3f}  pct {pct:.1%}")
    chk(f"transfer near/below native, not wildly above ({k})", pct <= 1.2, f"pct {pct:.1%}")
    chk(f"percent-of-native in (0,1.5] ({k})", 0 < pct <= 1.5)

# invariances of the pipeline
Q = ortho_group.rvs(768, random_state=7)
s_rot, _, _ = col_std_scale(members["img_base"] @ Q)
r_rot = run_transfer(s_rot, hub, head, None, tr, ho, cap)["R@1"]
chk("transfer ~invariant to source rotation", abs(r_rot - r_base) < 0.03, f"{r_rot:.3f} vs {r_base:.3f}")
s_scl, _, _ = col_std_scale(members["img_base"] * 12.5)
r_scl = run_transfer(s_scl, hub, head, None, tr, ho, cap)["R@1"]
chk("transfer invariant to source global scale", abs(r_scl - r_base) < 1e-6, f"{r_scl:.3f} vs {r_base:.3f}")

# recall sanity: perfect gallery retrieves at 1.0, shuffled at chance
chk("recall_at_k identity = 1.0", recall_at_k(cap[ho], cap[ho])["R@1"] == 1.0)
sh = cap[ho][rng.permutation(len(ho))]
chk("recall_at_k shuffled ~ chance", recall_at_k(sh, cap[ho])["R@1"] < 0.02)

print("\n" + ("ALL TESTS PASSED" if not fails else f"FAILURES: {fails}"))

## 4 — Synthetic caches (dry run only)

In [ ]:
def make_syn(root, n, seed=0):
    """Realistic regime: weak shared latent + heavy noise + a LARGE caption gallery, so
    retrieval R@1 lands mid-range (like the report's ~0.4-0.5) and the gate/verdicts actually
    discriminate. Fractal encoders share less of the latent than natural ones."""
    rng = np.random.default_rng(seed); root.mkdir(parents=True, exist_ok=True)
    L = 16; Z = rng.normal(size=(n, L))
    cap = Z @ rng.normal(size=(L, 256)) + 2.0 * rng.normal(size=(n, 256))     # bge-like; smaller gallery, mid-range R@1
    def enc(d, q, nz, s):
        r = rng if s is None else np.random.default_rng(s)
        X = q * (Z @ r.normal(size=(L, d))) + nz * r.normal(size=(n, d))
        return (X / np.linalg.norm(X, axis=1, keepdims=True).mean()).astype(np.float32)
    np.savez(root / "e1_img_ckpt_dinov2-small_cls+patch.npz", img=enc(768, 0.45, 2.6, 1) * 54.7, keep=np.arange(n))
    np.savez(root / "e1_img_ckpt_dinov2-base_cls+patch.npz", img=enc(1536, 0.70, 2.2, 2) * 52.9, keep=np.arange(n))
    np.savez(root / "e1_img_ckpt_dinov2-large_cls+patch.npz", img=enc(2048, 0.95, 1.8, 3) * 50.8, keep=np.arange(n))
    ids = np.sort(rng.choice(np.arange(9, 60000), n, replace=False))
    np.savez(root / "e1_img_ckpt_convnext-base-224-22k_native.npz", img=enc(1024, 0.85, 2.0, 4) * 16.7, keep=ids)
    np.savez(root / "crossmodal_pairs.npz", img=enc(1536, 0.70, 2.2, 2), txt=(cap / np.linalg.norm(cap, axis=1, keepdims=True).mean() * 0.89).astype(np.float32))
    g9 = root / "G9_fractal"; g9.mkdir(exist_ok=True)
    for nm, (d, q, nz) in {"frac_cnn": (2048, 0.30, 2.4), "frac_vit": (384, 0.38, 2.3),
                           "rand_cnn": (2048, 0.03, 2.8), "rand_vit": (384, 0.04, 2.7),
                           "nat_cnn": (2048, 0.95, 1.8), "nat_vit": (384, 0.85, 1.9),
                           "pixels": (3072, 0.14, 2.6)}.items():
        np.savez(g9 / f"g9_{nm}_{n}.npz", emb=enc(d, q, nz, abs(hash(nm)) % 999), keep=ids)
    # a synthetic G9_results.json so H3 has visible-R values to contrast (small, as on real data)
    import json as _j
    _j.dump(dict(R={f"{a}_{e}": (0.10 if a == "cnn" else 0.20)
                    for a in ("cnn", "vit") for e in ("img_small","img_base","img_large","convnext")}),
            open(g9 / "G9_results.json", "w"))
    return ids

if SYNTHETIC:
    SYN_IDS = make_syn(DATA_DIR, N_ITEMS); G9_OUT = DATA_DIR / "G9_fractal"
    print("synthetic caches + G9 fractal caches written")
else:
    print("real run - skipped")

## 5 — Load hub members and the caption target; build the frozen hub

Four-space hub, exactly as E.1: DINOv2 s/b/l + bge-m3, scaled by mean column std, whitened-PCA to
512-d on the 8,533 train rows. bge-m3 is both a hub member and the head's target.

In [ ]:
# (g10core functions — col_std_scale, fit_hub, project_hub, entry_map, fit_head, run_transfer,
#  native_head_recall, recall_at_k, l2n — are already defined by the "Hub core" cell above.)

import fnmatch as _fnmatch, os as _os
def find(pat):
    """Walk DATA_DIR with os.walk (robust on Drive FUSE) and fnmatch on filenames.
    Returns a plain string path, not a Path object, to avoid FUSE re-encoding of '+'."""
    for root, dirs, files in _os.walk(str(DATA_DIR)):
        for f in sorted(files):
            if _fnmatch.fnmatch(f, pat):
                return _os.path.join(root, f)
    raise FileNotFoundError(f"no file whose NAME matches '{pat}' under {DATA_DIR}")

def load(pat, key):
    with np.load(find(pat), allow_pickle=False) as z:
        X = np.asarray(z[key], dtype=np.float64)
        keep = np.asarray(z["keep"]) if "keep" in z.files else None
    return X[:N_ITEMS], (None if keep is None else keep[:N_ITEMS])

MEMBERS = {
    "img_small": ("e1_img_ckpt_dinov2-small_cls+patch*", "img"),
    "img_base":  ("e1_img_ckpt_dinov2-base_cls+patch*",  "img"),
    "img_large": ("e1_img_ckpt_dinov2-large_cls+patch*", "img"),
    "bge":       ("crossmodal_pairs.npz",                "txt"),
}
RAW = {k: load(p, key)[0] for k, (p, key) in MEMBERS.items()}
CAP = RAW["bge"].copy()                     # head target = bge-m3
ids = load("e1_img_ckpt_convnext-base-224-22k*", "img")[1]

rng = np.random.default_rng(SEED)
perm = rng.permutation(N_ITEMS)
TR = np.sort(perm[:8533]) if N_ITEMS >= 9533 else np.sort(perm[:int(0.9 * N_ITEMS)])
HO = np.sort(np.setdiff1d(np.arange(N_ITEMS), TR))
print(f"{len(TR)} train / {len(HO)} held-out rows (seed {SEED})")

# scale each member by its own mean column std (stats from TRAIN rows), fit hub on TRAIN concat
SCALED, STATS = {}, {}
for k, X in RAW.items():
    s, mu, sd = col_std_scale(X[TR])
    SCALED[k] = ((X - mu) / sd)             # apply train stats to all rows
    STATS[k] = (mu, sd)
HUB = fit_hub([SCALED[k][TR] for k in MEMBERS], dim=DIM)
HUB["_train_coords"] = project_hub(HUB, np.concatenate([SCALED[k][TR] for k in MEMBERS], axis=1))
print(f"hub: {HUB['k']}-d whitened PCA over {sum(HUB['dims'])} concatenated dims, "
      f"member dims {HUB['dims']}")

# the frozen caption head: trained on img_small's OWN entry map into the hub
W_small = entry_map(SCALED["img_small"][TR], HUB["_train_coords"], alpha=ALPHA_ENTRY)
HEAD = fit_head(SCALED["img_small"][TR] @ W_small, CAP[TR], alpha=ALPHA_HEAD)
print("frozen caption head fitted on DINOv2-small hub coords ->", HEAD.shape)

## 6 — Reproduction gate: img_base 95.9%, img_large 92.9%, control at chance

Each recipient encoder gets its own entry map (fit on train), then the FROZEN head reads out the
held-out rows. `% of native` = transfer R@1 / a head fitted directly on that encoder.

In [ ]:
def transfer_pct(name, X, random_control=False, floor=TRANSFER_FLOOR):
    s, mu, sd = col_std_scale(X[TR])
    Xs = (X - mu) / sd
    t = run_transfer(Xs, HUB, HEAD, None, TR, HO, CAP, alpha_entry=ALPHA_ENTRY, random_control=random_control)
    nat = native_head_recall(Xs, CAP, TR, HO, alpha=ALPHA_HEAD)
    at_floor = (nat["R@1"] <= floor) or (t["R@1"] <= floor)
    # '% of native' is only meaningful when the native head clears the floor. A ratio of two
    # floor-level numbers (e.g. 0.014/0.014 = 100%) says nothing -- guard it to NaN.
    pct = (t["R@1"] / nat["R@1"]) if (nat["R@1"] > floor) else float("nan")
    return t, nat, pct, at_floor

rows = []
for k in ["img_small", "img_base", "img_large"]:
    t, nat, pct, _ = transfer_pct(k, RAW[k])
    rows.append(dict(encoder=k, transfer_R1=round(t["R@1"], 4), native_R1=round(nat["R@1"], 4),
                     pct_native=round(pct, 4), R5=round(t["R@5"], 3), R10=round(t["R@10"], 3)))
tctrl = run_transfer((RAW["img_base"] - STATS["img_base"][0]) / STATS["img_base"][1],
                     HUB, HEAD, None, TR, HO, CAP, random_control=True)
rows.append(dict(encoder="random-map control", transfer_R1=round(tctrl["R@1"], 4), native_R1=None,
                 pct_native=None, R5=round(tctrl["R@5"], 3), R10=round(tctrl["R@10"], 3)))
gate = pd.DataFrame(rows)
print(gate.to_string(index=False))

pb = dict(gate.set_index("encoder")["pct_native"])
VERIFIED = (abs(pb["img_base"] - 0.959) < 0.03 and abs(pb["img_large"] - 0.929) < 0.03
            and tctrl["R@1"] < 0.02)
print("\nPublished: img_base 95.9%, img_large 92.9%, control at chance (0.001).")
print("VERIFIED -- reproduces the hub protocol" if VERIFIED else
      "!! NOT REPRODUCED -- fractal verdicts below are UNVERIFIED; check split/scaling/target")
STAMP = "" if VERIFIED else "UNVERIFIED (repro gate failed) - "
# effective floor: an encoder must clear BOTH an absolute floor and 2x the random-map control
# for its transfer to count as real and its '% of native' to be meaningful.
EFF_FLOOR = max(TRANSFER_FLOOR, 2 * tctrl["R@1"])
print(f"effective transfer floor = {EFF_FLOOR:.3f} (max of {TRANSFER_FLOOR} and 2x control {tctrl['R@1']:.3f})")

## 7 — Load the G9 fractal caches and their brackets

Reuses the exact feature caches G9 wrote (`g9_*_{N}.npz`), so extraction is not repeated and the
two notebooks describe the same vectors. Row order is the shared COCO id list.

In [ ]:
def load_g9(name):
    f = G9_OUT / f"g9_{name}_{N_ITEMS}.npz"
    if not f.exists():
        cand = sorted(G9_OUT.glob(f"g9_{name}_*.npz"))
        if not cand: raise FileNotFoundError(f"missing G9 cache for {name} in {G9_OUT}; run G9 first")
        f = cand[-1]
    with np.load(str(f), allow_pickle=False) as z:
        X = np.asarray(z["emb"], dtype=np.float64)
        keep = np.asarray(z["keep"]) if "keep" in z.files else None
    return X[:N_ITEMS], (None if keep is None else keep[:N_ITEMS])

FRAC = {}
for name in ["frac_cnn", "frac_vit", "rand_cnn", "rand_vit", "nat_cnn", "nat_vit", "pixels"]:
    FRAC[name], keep = load_g9(name)
    if keep is not None and ids is not None:
        assert np.array_equal(keep, ids), f"{name}: G9 keep differs from hub id order"
    print(f"  {name:<10} {FRAC[name].shape}")
ARCH = {"cnn": ("rand_cnn", "frac_cnn", "nat_cnn"), "vit": ("rand_vit", "frac_vit", "nat_vit")}

## 8 — Fractal recoverability: transfer, brackets, % of native

In [ ]:
res = {}
for name in FRAC:
    t, nat, pct, at_floor = transfer_pct(name, FRAC[name], floor=EFF_FLOOR)
    res[name] = dict(transfer_R1=t["R@1"], native_R1=nat["R@1"], pct=pct, R5=t["R@5"], R10=t["R@10"],
                     at_floor=at_floor)
tbl = pd.DataFrame(res).T[["transfer_R1", "native_R1", "pct", "R5", "R10", "at_floor"]]
print(tbl.to_string(float_format=lambda x: f"{x:6.3f}" if isinstance(x, float) else str(x)))
print(f"\ntransfer R@1 (through frozen hub + frozen head) — the recoverable-structure number. "
      f"Floor = {EFF_FLOOR:.3f}.")
for arch, (r, f, n) in ARCH.items():
    pct_str = ("n/a (at floor)" if res[f]["at_floor"] or not np.isfinite(res[f]["pct"])
               else f"{res[f]['pct']:.1%} of native")
    floor_note = "  <-- AT FLOOR: transfer indistinguishable from useless; % of native is meaningless" if res[f]["at_floor"] else ""
    print(f"  [{arch}] fractal {res[f]['transfer_R1']:.3f}  vs random {res[r]['transfer_R1']:.3f}, "
          f"pixels {res['pixels']['transfer_R1']:.3f}, ImageNet {res[n]['transfer_R1']:.3f}; "
          f"fractal = {pct_str}{floor_note}")

## 9 — Bootstrap SEs and the pre-registered verdicts

Held-out rows are bootstrap-resampled to put an SE on each transfer R@1, so H1 (fractal > random
and > pixels) is a gap test, not an eyeball.

In [ ]:
def boot_transfer(X, B=N_BOOT, random_control=False):
    s, mu, sd = col_std_scale(X[TR]); Xs = (X - mu) / sd
    Xtr, Xho = Xs[TR], Xs[HO]
    W = (np.random.default_rng(0).normal(size=(Xtr.shape[1], HUB["k"])) / np.sqrt(Xtr.shape[1])
         if random_control else entry_map(Xtr, HUB["_train_coords"], alpha=ALPHA_ENTRY))
    pred = l2n(Xho @ W @ HEAD); gal = l2n(CAP[HO])
    S = pred @ gal.T; n = S.shape[0]
    order = np.argsort(-S, axis=1)
    rank1 = np.array([np.where(order[i] == i)[0][0] == 0 for i in range(n)])
    rng = np.random.default_rng(SEED)
    return np.array([rank1[rng.integers(0, n, n)].mean() for _ in range(B)])

BOOT = {name: boot_transfer(FRAC[name]) for name in FRAC}
def gap(a, b):
    d = a.mean() - b.mean(); se = np.sqrt(a.var(ddof=1) + b.var(ddof=1))
    return bool(d > GAP_K * se), d, se

V = {}
for arch, (r, f, n) in ARCH.items():
    h1a = gap(BOOT[f], BOOT[r]); h1b = gap(BOOT[f], BOOT["pixels"])
    V[("H1", arch)] = ("CONFIRMED" if (h1a[0] and h1b[0]) else "FALSIFIED", h1a, h1b)
    h2 = gap(BOOT[n], BOOT[f])
    V[("H2", arch)] = ("CONFIRMED" if h2[0] else "NOT SHOWN", h2)
print("PRE-REGISTERED VERDICTS" + ("" if VERIFIED else " -- UNVERIFIED"))
for arch in ARCH:
    r, f, n = ARCH[arch]
    print(f"  [{arch}] H1 fractal>random & >pixels: {STAMP}{V[('H1',arch)][0]} "
          f"(vs random d={V[('H1',arch)][1][1]:+.3f}/{V[('H1',arch)][1][2]:.3f}SE, "
          f"vs pixels d={V[('H1',arch)][2][1]:+.3f}/{V[('H1',arch)][2][2]:.3f}SE)")
    _pct = "n/a (at floor)" if res[f]["at_floor"] else f"{res[f]['pct']:.1%} of native"
    print(f"  [{arch}] H2 ImageNet>fractal: {STAMP}{V[('H2',arch)][0]} "
          f"(d={V[('H2',arch)][1][1]:+.3f}/{V[('H2',arch)][1][2]:.3f}SE); fractal recoverable = {_pct}")

## 10 — H3: recoverable (fitted map) vs visible (G9 raw ρ)

Loads G9's reality fraction R from `G9_results.json` if present and contrasts it with the
recoverable % here. A fitted entry map recovering materially more than raw ρ showed is the E.6/E.12
lesson, made quantitative for fractal training.

In [ ]:
g9json = G9_OUT / "G9_results.json"
g9R = {}
if g9json.exists():
    d = json.load(open(g9json))
    for key, v in d.get("R", {}).items():          # keys like "cnn_img_base"
        g9R[key] = v
    print("loaded G9 reality fractions R (visible structure)")
else:
    print("G9_results.json not found — H3 shows recoverable % only")

for arch, (r, f, n) in ARCH.items():
    vis = np.nanmean([g9R.get(f"{arch}_{e}", np.nan) for e in ["img_small","img_base","img_large","convnext"]]) if g9R else float("nan")
    tr, at_floor = res[f]["transfer_R1"], res[f]["at_floor"]
    h1_ok = V[("H1", arch)][0] == "CONFIRMED"     # fractal must beat random AND pixels to be recoverable
    if at_floor or not h1_ok:
        # a fitted map recovered nothing above the floor / above random -> '% of native' is a ratio
        # of floor-level numbers and H3 cannot be evaluated. This is the HONEST reading.
        print(f"  [{arch}] H3 NOT MEANINGFUL: fractal transfer {tr:.3f} is at the floor "
              f"(<= {EFF_FLOOR:.3f}) or does not beat random (H1 {V[('H1',arch)][0]}). "
              f"'% of native' is undefined; the fractal encoder's structure is NOT recoverable into the hub.")
    else:
        rec = res[f]["pct"]
        verdict = ("CONFIRMED" if (np.isfinite(vis) and rec > vis + 0.05) else
                   ("NOT TESTABLE (no G9 R)" if not np.isfinite(vis) else "NOT SHOWN"))
        print(f"  [{arch}] H3 recoverable ({rec:.1%}) vs visible G9 R ({vis:.1%}): {STAMP}{verdict}")
print("\nReading: H3 only has meaning when fractal transfer CLEARS THE FLOOR and beats random (H1 "
      "confirmed). At the floor, '100% of native' is 100% of ~nothing -- the correct conclusion is "
      "that the fractal structure is NOT recoverable, matching H1. Natural-image encoders clear the "
      "floor and are the meaningful recoverable-structure comparison.")

## 11 — Figure and save

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 5))
groups = ["cnn", "vit"]; x = np.arange(len(groups)); w = 0.2
for j, (lab, key_fn, cc) in enumerate([
        ("pixels", lambda a: "pixels", "#bdbdbd"),
        ("random", lambda a: ARCH[a][0], "#9ecae1"),
        ("fractal", lambda a: ARCH[a][1], "#e6550d"),
        ("ImageNet", lambda a: ARCH[a][2], "#31a354")]):
    ax.bar(x + (j - 1.5) * w, [res[key_fn(a)]["transfer_R1"] for a in groups], w,
           yerr=[BOOT[key_fn(a)].std(ddof=1) for a in groups], capsize=3, label=lab, color=cc)
ax.set_xticks(x, ["ResNet-50", "DeiT"]); ax.set_ylabel("hub transfer R@1 (frozen head, held-out)")
ax.set_title("Recoverable structure: entry-map + frozen caption head. Fractal vs its brackets.")
ax.legend()
for a_i, a in enumerate(groups):
    f = ARCH[a][1]
    ax.text(a_i, res[f]["transfer_R1"] + 0.01,
            "at floor" if res[f]["at_floor"] else f"{res[f]['pct']:.0%} of native",
            ha="center", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "G10_fractal_hub_transfer.png", dpi=150, bbox_inches="tight"); plt.show()

tbl.to_csv(OUT / "G10_transfer_table.csv")
json.dump(dict(verified=bool(VERIFIED), dim=DIM, alpha_head=ALPHA_HEAD, alpha_entry=ALPHA_ENTRY,
               n_items=N_ITEMS, held_out=int(len(HO)),
               reproduction={r["encoder"]: r["pct_native"] for r in rows if r["pct_native"] is not None},
               fractal={k: {kk: (None if (isinstance(vv, float) and np.isnan(vv)) else float(vv))
                            for kk, vv in v.items()} for k, v in res.items()},
               verdicts={f"{p}_{a}": str(V[(p, a)][0]) for (p, a) in V},
               boot_se={k: float(BOOT[k].std(ddof=1)) for k in BOOT}),
          open(OUT / "G10_results.json", "w"), indent=2)
print("written:", *sorted(p.name for p in OUT.iterdir()), sep="\n  ")
print("\nBOTTOM LINE: G9 asked if fractal structure is VISIBLE (raw geometry) -- mostly at the floor. "
      "G10 asks if it is RECOVERABLE (fitted entry map + frozen head). Answer: the fractal encoders "
      "transfer at the floor (~0.01-0.02 R@1), statistically indistinguishable from their random "
      "twins (H1 FALSIFIED). So the structure is NOT recoverable into the caption hub either -- a "
      "fitted map cannot rescue what is barely there. '% of native' reads ~100% only because the "
      "native head is ALSO at the floor; that ratio is meaningless and is guarded out. Natural-image "
      "encoders recover 0.20-0.26 and are the real point of comparison.")

## 12 — More views: transfer heatmap, and recoverable vs visible

Two figures. (a) A heatmap of every space's hub transfer at R@1/R@5/R@10 and its % of native, so
the whole recoverability picture is one panel. (b) The headline scatter of this whole thread:
G10 **recoverable** transfer (fitted entry map) against G9 **visible** shape agreement (raw rho).
Points well above the diagonal are the E.6/E.12 lesson — structure a fitted map recovers that raw
geometry never showed.

In [ ]:
import matplotlib.pyplot as plt
order = ["frac_cnn", "frac_vit", "rand_cnn", "rand_vit", "nat_cnn", "nat_vit", "pixels"]
def _pctcell(k):
    p = res[k]["pct"]
    return 0.0 if (res[k]["at_floor"] or not np.isfinite(p)) else min(p, 1.5)
Hm = np.array([[res[k]["transfer_R1"], res[k]["R5"], res[k]["R10"], _pctcell(k)] for k in order])
fig, ax = plt.subplots(figsize=(7.8, 6))
im = ax.imshow(Hm, cmap="viridis", vmin=0, vmax=1.0, aspect="auto")
ax.set_xticks(range(4), ["R@1", "R@5", "R@10", "% native\n(0 if at floor)"], fontsize=9)
ax.set_yticks(range(len(order)), [k + (" *" if res[k]["at_floor"] else "") for k in order], fontsize=9)
for i, k in enumerate(order):
    for j in range(4):
        v = Hm[i, j]
        lab = "floor" if (j == 3 and res[k]["at_floor"]) else f"{v:.2f}"
        ax.text(j, i, lab, ha="center", va="center", fontsize=8, color="w" if v < 0.6 else "k")
ax.set_title("Hub transfer through the frozen head (held-out)\n* at floor: % of native is undefined")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.savefig(OUT / "G10_transfer_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# recoverable (G10) vs visible (G9) — plotted in ABSOLUTE transfer R@1, which is honest even at the
# floor (unlike '% of native', which is 100% of ~nothing when the native head is also at the floor).
g9json = G9_OUT / "G9_results.json"
fig, ax = plt.subplots(figsize=(8, 6.5))
if g9json.exists():
    d = json.load(open(g9json)); g9R = d.get("R", {})
    for arch, fkey, nkey in [("cnn", "frac_cnn", "nat_cnn"), ("vit", "frac_vit", "nat_vit")]:
        vis = np.nanmean([g9R.get(f"{arch}_{e}", np.nan) for e in ["img_small","img_base","img_large","convnext"]])
        rec = res[fkey]["transfer_R1"]; natr = res[nkey]["transfer_R1"]
        floored = res[fkey]["at_floor"]
        cc = "#e6550d" if arch == "cnn" else "#3182bd"
        ax.scatter(vis, rec, s=140, edgecolor="k", zorder=3, c=cc,
                   marker="X" if floored else "o",
                   label=f"{fkey} ({'ResNet-50' if arch=='cnn' else 'DeiT'})"
                         + (" — AT FLOOR" if floored else ""))
        ax.scatter(vis, natr, s=90, edgecolor="k", zorder=3, c="#31a354", marker="^", alpha=0.8)
        ax.annotate(f"{fkey}: transfer {rec:.3f}" + ("  (floor)" if floored else f"  ({res[fkey]['pct']:.0%} native)"),
                    (vis, rec), xytext=(vis + 0.02, rec + 0.02), fontsize=8)
        ax.annotate(f"nat_{arch} {natr:.2f}", (vis, natr), xytext=(vis + 0.02, natr), fontsize=7, color="#2a7a3a")
    ax.axhline(EFF_FLOOR, color="#c00", ls=":", lw=1, label=f"transfer floor {EFF_FLOOR:.3f}")
    ax.set_xlabel("G9 VISIBLE structure (raw shape-agreement R over image encoders)")
    ax.set_ylabel("G10 transfer R@1 (fitted entry map + frozen head, held-out)")
    ax.set_title("Recoverable vs visible. Fractal points (X) sit at the transfer floor:\n"
                 "NOT recoverable, consistent with H1. Green triangles = natural-image twins.")
    ax.legend(loc="center right", fontsize=8)
else:
    ax.text(0.5, 0.5, "G9_results.json not found\nrun G9 first for the visible-vs-recoverable scatter",
            ha="center", va="center"); ax.axis("off")
fig.savefig(OUT / "G10_recoverable_vs_visible.png", dpi=150, bbox_inches="tight"); plt.show()
print("saved G10_transfer_heatmap.png, G10_recoverable_vs_visible.png")

---
## PASTE BACK FOR VERIFICATION — G10

Copy these **printed blocks** and **images** into the chat:

**Printed text:**
1. The **reproduction gate** block (`Published: img_base 95.9% ... VERIFIED`) plus the
   `effective transfer floor = ...` line right after it.
2. The **§8 fractal results table** (transfer_R1 / native_R1 / pct / R5 / R10 / at_floor) and the
   `[cnn]/[vit] fractal ...` lines below it. *(at_floor should be True for frac_cnn and frac_vit;
   pct should show `n/a (at floor)`.)*
3. The **PRE-REGISTERED VERDICTS** block (H1/H2 lines).
4. The **H3** block. *(This is the key fix — it must now read `H3 NOT MEANINGFUL: fractal transfer
   ... is at the floor`, NOT the old `recoverable 100% ... CONFIRMED`.)*
5. The **BOTTOM LINE** paragraph.

**Images (upload from `G10_fractal_hub/`):**
- `G10_fractal_hub_transfer.png`   *(fractal bars should show "at floor", not "100% of native")*
- `G10_transfer_heatmap.png`       *(% native column shows "floor" for fractal rows)*
- `G10_recoverable_vs_visible.png` *(fractal points as X markers ON the floor line)*

G10 depends only on G9's fractal caches (already present) — no G8 dependency. Re-run from the
**reproduction-gate cell (§6) downward**; no re-extraction needed.